π0.5 companion lab：合成世界 + 单模型两阶段 / 共训练的公共内核。

这一篇在 Hi Robot 探针（`../hi-robot/`）的世界上加三个结构，分别对应 π0.5
（arXiv:2504.16054）的三件事：

1. **外观 → 语义的间接层**（对应 web 数据共训练）
   场景里的每个物体是一个**类型** t ∈ 0..T-1，观测给的是外观向量 φ_t，
   真值决策用的是语义属性 z_t ∈ {±1}^M。模型要先学一个槽编码器
   ψ: φ → ẑ，才能做「按属性筛选」的技能选择。web 数据（WD）直接给
   (φ_t, z_t) 对——**包括机器人演示里没出现过的类型**（web-only 类型）。
   没有 WD 时，ψ 只能从技能标签里间接学，web-only 类型的语义是瞎猜的。

2. **环境上下文向量**（对应多环境 scaling）
   每个环境 e 有一个上下文 κ_e ∈ R^4，动作里带一个环境相关项
   (v·x)·u(κ_e)，u 是一个**固定的非线性映射**（κ 进观测——新家是看得见的，
   u(·) 要从数据里学）。环境数少时 u(·) 只能背下来那几个 κ，环境数多了
   才能泛化到新 κ——这就是论文 Fig.8 那条 3→104 曲线的合成对应物。

3. **implicit HL 对照**（对应论文 Fig.13）
   「共训练时带上子任务标签、部署时只跑低层」——动作头直接条件于原始
   约束而不是预测出的技能。对照：joint（两阶段推理）/ implicit / flat
   （连子任务监督都没有）。

符号沿用 04 篇：约束 c 是 M×3 逐位 one-hot，技能 s* 由 W_g 打分选出，
动作 a* = μ_s + B x_z + 环境项 + η。

In [ ]:
import numpy as np

# ---------------------------------------------------------------- 世界常量
M = 4            # 每个类型的语义属性位数（±1）
T = 10           # 物体类型数：7 个「机器人演示里出现过」+ 3 个「web-only」
K = 5            # 每个场景的槽位数
S = 6            # 技能数
D_A = 8          # 动作维度
DONE = S - 1     # 「收工」伪技能
D_C = M * 3      # 约束 one-hot
D_PHI = 8        # 外观向量维度（观测给这个，不给 z）
D_X = K * (M + 1)   # 语义场景向量：每槽 [z (M), done (1)]
D_CTX = 4        # 环境上下文维度
N_SEEN = 7       # 机器人演示覆盖的类型数
HID = 64         # 各网络隐藏宽度
SIG_ETA = 0.15   # 不可预测噪声

N_MM_ENV = 8     # MM 数据的环境数（约 400 小时 / 目标平台的量级）
N_ME_ENV = 60    # ME 数据额外带来的环境数
N_TEST_ENV = 6   # 测试环境数

### 几何小工具：正交行、正单纯形、属性枚举

In [ ]:
def _orthonormal(rng, n, d):
    """n 个 d 维单位正交行（n ≤ d）。"""
    Q, _ = np.linalg.qr(rng.normal(size=(d, d)))
    return Q[:, :n].T


def _simplex(k: int, d: int) -> np.ndarray:
    """k 行正单纯形（k = d+1）：范数相同、两两内积相同、和为 0。"""
    C = np.eye(k) - np.full((k, k), 1.0 / k)
    ev, U = np.linalg.eigh(C)
    return U[:, -d:] * np.sqrt(ev[-d:])


def _all_z(m: int = M) -> np.ndarray:
    """枚举 {±1}^m 的全部 2^m 个属性模式。"""
    g = np.array(np.meshgrid(*[[-1.0, 1.0]] * m, indexing="ij")).reshape(m, -1).T
    return g

### World：合成世界（类型语义 / 环境上下文 / 真值 / 评分）

In [ ]:
class World:
    def __init__(self, seed: int = 0):
        rng = np.random.default_rng(seed)
        self.rng = rng

        # ---- 类型与语义：7 个 seen 类型 + 3 个 web-only 类型，属性模式互不相同
        patterns = _all_z()
        pick = rng.permutation(len(patterns))[:T]
        self.z_types = patterns[pick]                     # (T, M)
        self.seen_types = np.arange(N_SEEN)
        self.web_types = np.arange(N_SEEN, T)
        # 外观：10 个彼此拉开距离的随机单位向量（ψ 要从 φ 恢复 z）
        phi = rng.normal(size=(T, D_PHI))
        phi /= np.linalg.norm(phi, axis=1, keepdims=True)
        self.phi_types = phi

        # ---- 动作内核（同 04 篇）
        self.B = rng.normal(size=(D_A, D_X))
        self._calibrate_B()
        self.mu = _orthonormal(rng, S, D_A)
        self.W_g = self._balanced_scorer(rng)

        # ---- 环境上下文：u(κ) = Cκ + tanh(κ W₂ᵀ) C₂（带固定的非线性分量，
        #      让「从很少几个环境学 u(·)」真的学不出来），动作环境项 = (v·x)·u
        self.C = rng.normal(size=(D_A, D_CTX))
        self.W2 = rng.normal(size=(16, D_CTX))
        self.C2 = rng.normal(size=(D_A, 16))
        self.v = rng.normal(size=D_X)
        self.v /= np.linalg.norm(self.v)
        self._calibrate_C()

        # ---- 约束池（同 04 篇）
        self.single = [((j,), ()) for j in range(M)] + [((), (j,)) for j in range(M)]
        self.composite = [((j,), ((j + 1) % M,)) for j in range(M)]
        order = rng.permutation(len(self.single))
        self.train_pool = [self.single[i] for i in order]

    # ------------------------------------------------------------- 类型语义
    def _balanced_scorer(self, rng, tries: int = 4000):
        """W_g：单纯形 + 旋转，要求 argmax 在 {±1}^M 上分布均衡（同 04 篇）。"""
        base = _simplex(S - 1, M)
        grid = _all_z()
        best, best_gap = None, -1.0
        for _ in range(tries):
            Q, _ = np.linalg.qr(rng.normal(size=(M, M)))
            W = base @ Q
            cnt = np.bincount(np.argmax(grid @ W.T, axis=1), minlength=S - 1)
            p = cnt / cnt.sum()
            if p.min() > best_gap:
                best, best_gap = W, p.min()
            if p.min() >= 0.15 and p.max() <= 0.30:
                return W
        return best

    def _calibrate_B(self):
        """缩放 B 使 E‖Bx‖² = 0.25（逐维平均口径，目标除以 D_A——04 篇踩过的坑）。"""
        x = self.sample_semantic(8192, np.arange(T))[0]
        cur = float(np.mean(np.square(x @ self.B.T)))
        self.B *= np.sqrt((0.25 / D_A) / max(cur, 1e-12))

    def env_u(self, kap: np.ndarray) -> np.ndarray:
        """环境项的方向向量 u(κ) = Cκ + tanh(κ W₂ᵀ) C₂（固定的非线性映射）。"""
        return kap @ self.C.T + np.tanh(kap @ self.W2.T) @ self.C2.T

    def _calibrate_C(self):
        """缩放 (C, C2) 使环境项 E‖(v·x)·u(κ)‖² = 0.25（与 Bx 项同量级）。"""
        x = self.sample_semantic(8192, np.arange(T))[0]
        kap = self.rng.normal(size=(len(x), D_CTX))
        env = (x @ self.v)[:, None] * self.env_u(kap)
        cur = float(np.mean(np.square(env)))
        sc = np.sqrt((0.25 / D_A) / max(cur, 1e-12))
        self.C *= sc
        self.C2 *= sc

    # ------------------------------------------------------------- 场景采样
    def sample_scene(self, n: int, types, rng=None):
        """采 n 个场景。返回 (z (n,K,M), types (n,K), done (n,K), 槽位类型表)。"""
        g = self.rng if rng is None else rng
        ty = g.choice(np.asarray(types), size=(n, K), replace=True)
        z = self.z_types[ty]                              # (n, K, M)
        p_done = g.uniform(0.0, 0.6, size=(n, 1))
        done = np.where(g.random((n, K)) < p_done, 1.0, -1.0)
        return z, ty, done

    def semantic_vec(self, z, done) -> np.ndarray:
        """(z, done) → 语义场景向量 x（模型与真值共用的表示）。"""
        n = z.shape[0]
        return np.concatenate([z.reshape(n, K * M), done.reshape(n, K)], axis=1)

    def sample_semantic(self, n: int, types, rng=None):
        """直接返回语义场景 x（D 组 / 标定用，跳过外观）。"""
        z, _, done = self.sample_scene(n, types, rng)
        return self.semantic_vec(z, done), z, done

    def observe(self, ty: np.ndarray) -> np.ndarray:
        """槽位类型表 → 外观观测 (n, K, D_PHI)。"""
        return self.phi_types[ty]

    # ------------------------------------------------------------- 真值
    def target(self, x: np.ndarray, r, rbar) -> np.ndarray:
        """满足约束且未处理的、下标最小的槽位；没有则 −1。"""
        t = np.atleast_2d(x).reshape(-1, K, M + 1)
        z, done = t[:, :, :M], t[:, :, M]
        ok = np.ones(z.shape[:2], dtype=bool)
        for j in r:
            ok &= z[:, :, j] > 0
        for j in rbar:
            ok &= z[:, :, j] < 0
        ok &= done < 0
        has = ok.any(axis=1)
        return np.where(has, np.argmax(ok, axis=1), -1)

    def skill(self, x: np.ndarray, r, rbar) -> np.ndarray:
        return self.skill_vec(x, self.cvec(r, rbar)[None, :])[0]

    def skill_vec(self, x: np.ndarray, cv: np.ndarray) -> np.ndarray:
        """向量化真值技能：cv (n, M*3) 逐位 one-hot，每个样本一个约束。"""
        t = np.atleast_2d(x).reshape(-1, K, M + 1)
        z, done = t[:, :, :M], t[:, :, M]
        c3 = cv.reshape(-1, M, 3)
        r1, r0 = c3[:, :, 0], c3[:, :, 1]                 # (n, M)
        ok1 = (r1[:, None, :] > 0) & (z > 0)
        ok0 = (r0[:, None, :] > 0) & (z < 0)
        off = (r1[:, None, :] == 0) & (r0[:, None, :] == 0)
        ok = (ok1 | ok0 | off).all(axis=2)                # (n, K)
        ok &= done < 0
        has = ok.any(axis=1)
        idx = np.where(has, np.argmax(ok, axis=1), 0)
        zt = z[np.arange(len(z)), idx]                    # (n, M)
        sk = np.argmax(zt @ self.W_g.T, axis=-1)
        return np.where(has, sk, DONE)

    def action(self, x: np.ndarray, s: np.ndarray, kap: np.ndarray,
               *, noise: bool = True, rng=None) -> np.ndarray:
        """真值动作 a* = μ_s + Bx + (v·x)·u(κ) + η。"""
        x = np.atleast_2d(x)
        kap = np.atleast_2d(kap)
        env = (x @ self.v)[:, None] * self.env_u(kap)
        a = self.mu[np.asarray(s).reshape(-1)] + x @ self.B.T + env
        if noise:
            g = self.rng if rng is None else rng
            a = a + SIG_ETA * g.normal(size=a.shape)
        return a

    def oracle_env_gap(self, x: np.ndarray, kap: np.ndarray) -> float:
        """环境项的相对量级（给「完全不看 κ 的模型」当前场景的误差下界）。"""
        x = np.atleast_2d(x)
        kap = np.atleast_2d(kap)
        env = (x @ self.v)[:, None] * self.env_u(kap)
        return float(np.mean(np.square(env)))

    def rel_err(self, a_hat, a_star) -> float:
        num = float(np.mean(np.square(a_hat - a_star)))
        den = float(np.mean(np.square(a_star - a_star.mean(axis=0))))
        return num / max(den, 1e-12)

    def cvec(self, r, rbar) -> np.ndarray:
        v = np.zeros((M, 3))
        rset, rbset = set(r), set(rbar)
        for j in range(M):
            v[j, 0 if j in rset else (1 if j in rbset else 2)] = 1.0
        return v.reshape(-1)

    def cvec_batch(self, cs) -> np.ndarray:
        return np.stack([self.cvec(*c) for c in cs])

### 数据批：向量化采样

In [ ]:
# ---------------------------------------------------------------- 数据批
def make_batch(world: World, cs, types, kaps, n_per: int, rng):
    """混合采样一批 (约束, 场景, κ)，全部向量化。

    返回 dict：cv (n,D_C)、obs (n,K,D_PHI)、x (n,D_X)、done (n,K)、
    kap (n,D_CTX)、s* (n,)、a* (n,D_A)。每个样本独立抽约束、类型与 κ。
    """
    n = len(cs) * n_per
    ty = rng.choice(np.asarray(types), size=(n, K), replace=True)
    z = world.z_types[ty]
    p_done = rng.uniform(0.0, 0.6, size=(n, 1))
    done = np.where(rng.random((n, K)) < p_done, 1.0, -1.0)
    x = world.semantic_vec(z, done)
    ci = rng.integers(0, len(cs), size=n)
    cv = world.cvec_batch([cs[i] for i in ci])
    ki = rng.integers(0, len(kaps), size=n)
    kap = kaps[ki]
    s = world.skill_vec(x, cv)
    a = world.action(x, s, kap, noise=True, rng=rng)
    return dict(cv=cv, obs=world.observe(ty), x=x, done=done, ty=ty,
                kap=kap, s=s, a=a)

### 小网络：前向、手写反向、Adam

In [ ]:
# ---------------------------------------------------------------- 小网络
def init_mlp(din: int, dout: int, rng, hid: int = HID):
    s1 = np.sqrt(2.0 / din)
    s2 = np.sqrt(1.0 / hid)
    return [rng.normal(0, s1, (din, hid)), np.zeros(hid),
            rng.normal(0, s2, (hid, dout)), np.zeros(dout)]


def fwd(p, X):
    H = np.tanh(X @ p[0] + p[1])
    return H, H @ p[2] + p[3]


def bwd(p, X, H, dY):
    """参数梯度。约定：dY 已经是 dL/dY（含全部归一化），这里不再除以 n。

    （hirobot_lib 的版本在 bwd 里再除一次 n，恰好被「批大小 = 动作维度」
    的自校验批次掩盖；本篇批次 128 ≠ 动作维度 8，必须把口径分开。）
    """
    dH = (dY @ p[2].T) * (1.0 - H * H)
    return [X.T @ dH, dH.sum(axis=0), H.T @ dY, dY.sum(axis=0)]


def mse_grad(out, target):
    """损失 = 每样本向量范数平方、对 batch 取平均（flow matching 的 ‖·‖² 口径）。

    这样打印的损失与 dL/dout 严格对应：d = 2(out-target)/n。
    （np.mean(np.square(...)) 会对输出维也取平均，和这个梯度差一个 D_A。）
    """
    n = out.shape[0]
    diff = out - target
    loss = float(np.mean(np.sum(np.square(diff), axis=-1)))
    return loss, 2.0 * diff / n


def softmax_ce(logits, y):
    n = logits.shape[0]
    z = logits - logits.max(axis=-1, keepdims=True)
    e = np.exp(z)
    pr = e / e.sum(axis=-1, keepdims=True)
    loss = float(-np.mean(np.log(pr[np.arange(n), y] + 1e-12)))
    d = pr.copy()
    d[np.arange(n), y] -= 1.0
    return loss, d / n


class Adam:
    def __init__(self, params, lr: float = 3e-3):
        self.t = 0
        self.lr = lr
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]

    def step(self, params, grads):
        self.t += 1
        b1, b2, eps = 0.9, 0.999, 1e-8
        for i, (p, g) in enumerate(zip(params, grads, strict=True)):
            self.m[i] = b1 * self.m[i] + (1 - b1) * g
            self.v[i] = b2 * self.v[i] + (1 - b2) * g * g
            p -= self.lr * (self.m[i] / (1 - b1 ** self.t)) / (
                np.sqrt(self.v[i] / (1 - b2 ** self.t)) + eps)


def onehot(idx: np.ndarray, n: int) -> np.ndarray:
    o = np.zeros((len(idx), n))
    o[np.arange(len(idx)), np.asarray(idx)] = 1.0
    return o

### 模型：单模型两阶段的抽象

In [ ]:
# ---------------------------------------------------------------- 模型
class Pi05Net:
    """单模型两阶段：槽编码器 ψ（语义）+ 共享主干 g + 技能头 + 动作头。

    变体开关
    --------
    use_skill_input : 动作头是否条件于技能（False = implicit / flat 的
        「只跑低层」形态，动作头直接看约束）。
    train_skill     : 是否训练技能头（False = flat，连子任务监督都没有）。
    """

    def __init__(self, world: World, rng, *, use_skill_input: bool = True,
                 train_skill: bool = True):
        self.w = world
        self.use_skill_input = use_skill_input
        self.train_skill = train_skill
        self.psi = init_mlp(D_PHI, M, rng)                     # 外观 → 语义
        self.trunk = init_mlp(D_C + D_X, HID, rng)
        self.skill_head = init_mlp(HID, S, rng) if train_skill else None
        din = HID + (S if use_skill_input else D_C) + D_CTX
        self.act_head = init_mlp(din, D_A, rng)
        self.params = self.psi + self.trunk + self.act_head
        if self.skill_head is not None:
            self.params = self.params + self.skill_head

    # ---- 前向
    def zhat(self, obs):
        """外观 → 每槽语义 ẑ（tanh 压到 ±1）。"""
        n, kk, _ = obs.shape
        _, out = fwd(self.psi, obs.reshape(n * kk, D_PHI))
        return np.tanh(out).reshape(n, kk, M)

    def scene_rep(self, obs, done):
        """(obs, done) → 模型视角的语义场景向量。"""
        n = obs.shape[0]
        return np.concatenate([self.zhat(obs).reshape(n, K * M),
                               done.reshape(n, K)], axis=1)

    def features(self, cv, xs):
        _, h = fwd(self.trunk, np.concatenate([cv, xs], axis=1))
        return h

    def skill_logits(self, feat):
        _, out = fwd(self.skill_head, feat)
        return out

    def act_out(self, feat, cv, xs, kap, s):
        if self.use_skill_input:
            z_in = np.concatenate([feat, onehot(s, S), kap], axis=1)
        else:
            z_in = np.concatenate([feat, cv, kap], axis=1)
        _, out = fwd(self.act_head, z_in)
        return out

## 跑实验：公共件

In [ ]:
# -*- coding: utf-8 -*-
"""π0.5 companion lab：共训练 / 单模型两阶段的合成探针。

跑法（数字要用 miniconda python 生成，numpy 2.1.1；managed python 的 numpy
2.5.x 会给出略不同的数值）::

    ~/Software/miniconda3/bin/python pi05_lab.py

四组实验（断言都在组内，最后汇总断言总数）::

    自校验   手写反向 vs 有限差分；世界不变量
    A 组     共训练配方消融（WD / ME / CE）——对应论文 Fig.10/11
    B 组     环境数量的 scaling——对应论文 Fig.8 的 3→104 曲线
    C 组     joint vs implicit HL vs flat——对应论文 Fig.13
    D 组     状态进 prompt 的量化代价——对应 openpi tokenizer.py 的 π0.5 格式
"""


import numpy as np


N_STEPS = 2500
BATCH = 128
N_SEEDS = 3

_ASSERTS: list[str] = []


def check(name: str, ok: bool, detail: str = "") -> None:
    tag = "[ok]" if ok else "[FAIL]"
    _ASSERTS.append(f"{tag} {name}" + (f"：{detail}" if detail else ""))
    print(f"  {tag} {name}" + (f"（{detail}）" if detail else ""))


def banner(title: str) -> None:
    line = "=" * 74
    print(f"\n{line}\n{title}\n{line}")


# ---------------------------------------------------------------- 训练一步
def _zero_grads(net: Pi05Net):
    return [np.zeros_like(p) for p in net.params]


def _psi_ix():
    return slice(0, 4)


def _trunk_ix():
    return slice(4, 8)


def _act_ix():
    return slice(8, 12)


def _skill_ix(net: Pi05Net):
    return slice(12, 16) if net.skill_head is not None else slice(0, 0)


def train_batch_grads(net: Pi05Net, b, *, with_skill: bool, with_act: bool,
                      alpha: float = 1.0):
    """一个机器人数据批的反向。返回 (losses, grads)。"""
    g = _zero_grads(net)
    obs, done = b["obs"], b["done"]
    n, kk, _ = obs.shape
    cv, kap, s, a = b["cv"], b["kap"], b["s"], b["a"]

    # ---- psi 前向（每槽独立）
    X1 = obs.reshape(n * kk, D_PHI)
    H1, P1 = fwd(net.psi, X1)
    zh = np.tanh(P1).reshape(n, kk * M)
    xs = np.concatenate([zh, done], axis=1)

    # ---- trunk 前向
    Xg = np.concatenate([cv, xs], axis=1)
    H2, F2 = fwd(net.trunk, Xg)
    dF2 = np.zeros_like(F2)
    losses = {}

    # ---- 技能头（CE）
    if with_skill and net.skill_head is not None:
        H3, L3 = fwd(net.skill_head, F2)
        loss_ce, dL = softmax_ce(L3, s)
        losses["ce"] = loss_ce
        gk = bwd(net.skill_head, F2, H3, dL)
        for i in range(4):
            g[12 + i] += gk[i]
        # 经技能头隐藏层回传到 F2（漏掉 tanh' 与 W1 是上一版的 bug）
        dH3 = (dL @ net.skill_head[2].T) * (1.0 - H3 * H3)
        dF2 += dH3 @ net.skill_head[0].T

    # ---- 动作头（MSE，teacher forcing：训练时条件于真值技能）
    if with_act:
        zin = (np.concatenate([F2, onehot(s, S), kap], axis=1)
               if net.use_skill_input else
               np.concatenate([F2, cv, kap], axis=1))
        H4, A4 = fwd(net.act_head, zin)
        loss_mse, dA = mse_grad(A4, a)
        losses["mse"] = loss_mse
        gk = bwd(net.act_head, zin, H4, alpha * dA)
        for i in range(4):
            g[8 + i] += gk[i]
        # 经动作头隐藏层回传到 F2（cv / κ 是输入，不回传）
        dH4 = (alpha * dA @ net.act_head[2].T) * (1.0 - H4 * H4)
        dF2 += (dH4 @ net.act_head[0].T)[:, :HID]

    # ---- trunk 反向（参数梯度 + 输入梯度，后者继续传给 ψ）
    gk = bwd(net.trunk, Xg, H2, dF2)
    for i in range(4):
        g[4 + i] += gk[i]
    dH2 = (dF2 @ net.trunk[2].T) * (1.0 - H2 * H2)
    dxs = (dH2 @ net.trunk[0].T)[:, D_C:]   # done 列是输入，不回传

    # ---- psi 反向（经 tanh；done 列是输入，不回传）
    dz = dxs[:, :K * M].reshape(n, kk, M)
    dP1 = dz * (1.0 - np.tanh(P1.reshape(n, kk, M)) ** 2)
    gk = bwd(net.psi, X1, H1, dP1.reshape(n * kk, M))
    for i in range(4):
        g[0 + i] += gk[i]
    return losses, g


def wd_batch_grads(net: Pi05Net, world: World, rng, n: int = 64):
    """web 数据批：直接监督 ψ(φ_t) → z_t（含 web-only 类型）。"""
    idx = rng.integers(0, T, size=n)
    phi = world.phi_types[idx]
    zt = world.z_types[idx]
    H1, P1 = fwd(net.psi, phi)
    out = np.tanh(P1)
    loss, dA = mse_grad(out, zt)
    g = _zero_grads(net)
    dP1 = dA * (1.0 - out ** 2)
    gk = bwd(net.psi, phi, H1, dP1)
    for i in range(4):
        g[i] += gk[i]
    return loss, g


def wd_qa_grads(net: Pi05Net, world: World, rng, n: int = 96):
    """web 语义问答：对**全部**属性模式（含 web-only 类型）问「该选哪个技能」。

    这是纯语言/语义监督，不含任何机器人动作：它走 ψ → 共享 trunk → 技能头，
    对应 web 数据给 VLM 主干补语义、而不是补动作。复合约束仍不进训练集。
    """
    ty = rng.integers(0, T, size=(n, K))
    z = world.z_types[ty]
    done = -np.ones((n, K))
    x = world.semantic_vec(z, done)
    ci = rng.integers(0, len(world.train_pool), size=n)
    cv = world.cvec_batch([world.train_pool[i] for i in ci])
    s = world.skill_vec(x, cv)
    b = dict(cv=cv, obs=world.observe(ty), x=x, done=done, ty=ty,
             kap=np.zeros((n, D_CTX)), s=s,
             a=np.zeros((n, D_A)))
    return train_batch_grads(net, b, with_skill=True, with_act=False)[1]


def train(world: World, seed: int, *, use_wd=True, use_ce=True, use_me=True,
          use_skill_input=True, train_skill=True, n_steps=N_STEPS,
          kaps_mm=None, kaps_me=None, kaps_ce=None, verbose=False):
    """训练一个变体。环境池由调用方显式给出（保证各变体评的是同一批环境）。"""
    rng = np.random.default_rng(seed)
    net = Pi05Net(world, rng, use_skill_input=use_skill_input,
                  train_skill=train_skill)
    opt = Adam(net.params, lr=3e-3)
    if kaps_mm is None:
        kaps_mm = world.rng.normal(size=(N_MM_ENV, D_CTX))
    if kaps_me is None:
        kaps_me = world.rng.normal(size=(N_ME_ENV, D_CTX)) if use_me else \
            np.zeros((0, D_CTX))
    kaps_me = kaps_me if use_me else np.zeros((0, D_CTX))
    kaps_all = np.concatenate([kaps_mm, kaps_me], axis=0) if len(kaps_me) \
        else kaps_mm
    if kaps_ce is None:
        kaps_ce = world.rng.normal(size=(20, D_CTX))
    ce_cs = world.train_pool        # 复合约束始终留作测试，不进任何训练集
    task = rng.random(n_steps)
    for t in range(n_steps):
        g = _zero_grads(net)
        # MM 批（总是有）
        r = task[t]
        pool = kaps_all if r < 0.35 and use_me else kaps_mm
        b = make_batch(world, world.train_pool, world.seen_types, pool,
                       max(1, BATCH // len(world.train_pool)), rng)
        _, gmm = train_batch_grads(net, b, with_skill=train_skill,
                                   with_act=True)
        for i in range(len(g)):
            g[i] += gmm[i] * 0.5
        # CE 批（更多样的任务与环境，只喂技能头）
        if use_ce and 0.35 <= r < 0.65:
            b = make_batch(world, ce_cs, world.seen_types, kaps_ce,
                           max(1, BATCH // len(ce_cs)), rng)
            _, gce = train_batch_grads(net, b, with_skill=True, with_act=False)
            for i in range(len(g)):
                g[i] += gce[i] * 0.5
        # WD 批：一半直接监督 ψ(φ→z)，一半是覆盖全部属性模式的语义问答
        if use_wd and r >= 0.65:
            if r >= 0.825:
                _, gwd = wd_batch_grads(net, world, rng)
                for i in range(4):
                    g[i] += gwd[i]
            else:
                gqa = wd_qa_grads(net, world, rng)
                for i in range(len(g)):
                    g[i] += gqa[i]
        opt.step(net.params, g)
        if verbose and (t + 1) % 500 == 0:
            print(f"    step {t+1}")
    return net


# ---------------------------------------------------------------- 评测
def eval_split(net: Pi05Net, world: World, cs, types, kaps, n_per, rng,
               web_only_target: bool | None = None):
    """一个评测切分。返回 (技能准确率, 端到端相对误差, 真值技能相对误差, n)。

    web_only_target=True 时只统计「目标槽是 web-only 类型」的样本。
    """
    accs, errs, oerrs, tot = [], [], [], 0
    for c in cs:
        b = make_batch(world, [c], types, kaps, n_per, rng)
        xs_hat = net.scene_rep(b["obs"], b["done"])
        feat = net.features(b["cv"], xs_hat)
        has_skill = net.use_skill_input and net.skill_head is not None
        if has_skill:
            s_hat = np.argmax(net.skill_logits(feat), axis=1)
        else:
            s_hat = b["s"]          # implicit/flat 没有技能输出，准确率无意义
        a_hat = net.act_out(feat, b["cv"], xs_hat, b["kap"], s_hat)
        a_star = b["a"]
        idx = world.target(b["x"], *c)
        m = np.ones(len(idx), dtype=bool)
        if web_only_target is not None:
            tslot = np.where(idx >= 0, b["ty"][np.arange(len(idx)), idx], -1)
            is_web = np.isin(tslot, world.web_types)
            m = is_web if web_only_target else ~is_web
        if not m.any():
            continue
        accs.append(float(np.mean(s_hat[m] == b["s"][m])) if has_skill
                    else float("nan"))
        errs.append(world.rel_err(a_hat[m], a_star[m]))
        # 真值技能条件下的模型输出（隔离「做什么」的错误，只看「怎么做」）
        a_or = net.act_out(feat, b["cv"], xs_hat, b["kap"], b["s"])
        oerrs.append(world.rel_err(a_or[m], a_star[m]))
        tot += int(m.sum())
    if not accs:
        return float("nan"), float("nan"), float("nan"), 0
    return float(np.mean(accs)), float(np.mean(errs)), float(np.mean(oerrs)), tot


def run_variant(world: World, seed: int, pools, **kw):
    """pools = dict(mm, me, ce, test)：训练与评测用同一批环境。"""
    net = train(world, seed, kaps_mm=pools["mm"], kaps_me=pools["me"],
                kaps_ce=pools["ce"], **kw)
    rng = np.random.default_rng(1000 + seed)
    out = {}
    out["id"] = eval_split(net, world, world.train_pool, world.seen_types,
                           pools["mm"], 24, rng)
    out["env"] = eval_split(net, world, world.train_pool, world.seen_types,
                            pools["test"], 24, rng)
    out["oodc"] = eval_split(net, world, world.composite, world.seen_types,
                             pools["mm"], 24, rng)
    out["oodt"] = eval_split(net, world, world.composite, np.arange(T),
                             pools["mm"], 120, rng, web_only_target=True)
    out["seen_t"] = eval_split(net, world, world.composite, np.arange(T),
                               pools["mm"], 120, rng, web_only_target=False)
    return out


def avg_over_seeds(world: World, seeds, pools, **kw):
    runs = [run_variant(world, s, pools, **kw) for s in seeds]
    keys = runs[0].keys()
    out = {}
    for k in keys:
        out[k] = tuple(float(np.mean([r[k][j] for r in runs]))
                       for j in range(3))
    return out


# ---------------------------------------------------------------- main
def _gap_args(world: World):
    x = world.sample_semantic(8192, np.arange(T))[0]
    kap = world.rng.normal(size=(len(x), D_CTX))
    return x, kap


def state_quant_probe(bins_list=(8, 16, 32, 64, 256), n=200000, seed=11):
    """π0.5 把本体状态离散化进 prompt（openpi tokenizer.py L26-28）。

    状态 18 维（论文：18–19 DoF）、取值 [-1,1]；动作 a = μ_s + B_s x + η
    （μ 是正交的技能原型，同 04 篇的设定）。量化误差
    E‖B_s(x−x̂)‖² / Var(a) 随 bins 变化；噪声下界 = σ²/Var(a)。
    """
    rng = np.random.default_rng(seed)
    D_S = 18
    B = rng.normal(size=(D_A, D_S))
    x = rng.uniform(-1, 1, size=(n, D_S))
    cur = float(np.mean(np.square(x @ B.T)))
    B *= np.sqrt((0.25 / D_A) / max(cur, 1e-12))
    mu = _orthonormal(rng, S, D_A)
    s = rng.integers(0, S, size=n)
    a = mu[s] + x @ B.T
    a_obs = a + 0.15 * rng.normal(size=(n, D_A))
    var = float(np.mean(np.square(a_obs - a_obs.mean(axis=0))))
    out = {"floor": 0.15 ** 2 / var}
    for bins in bins_list:
        xq = np.round((x + 1) / 2 * (bins - 1)) / (bins - 1) * 2 - 1
        aq = mu[s] + xq @ B.T
        out[bins] = float(np.mean(np.square(aq - a))) / var
    return out

**世界与自校验**

In [ ]:
world = World(0)
seeds = list(range(N_SEEDS))

# ------------------------------------------------------------ 自校验

**A 组：共训练配方消融**

In [ ]:
banner("自校验：手写反向 vs 有限差分；世界不变量")
rng = np.random.default_rng(7)
net = Pi05Net(world, rng)
b = make_batch(world, world.train_pool, world.seen_types,
               world.rng.normal(size=(4, D_CTX)), 4, rng)
# ---- 自校验统一用「随机方向的方向导数」：数值稳定，也免去逐元素差分
def dir_deriv(net, P, j, with_skill, with_act, eps=1e-6, seed=3):
    d = np.random.default_rng(seed).normal(size=P[j].shape)
    d /= np.linalg.norm(d)
    # 必须就地写回：net.params[j] = ... 只换列表项，net.psi 等仍指旧数组
    np.copyto(net.params[j], P[j] + eps * d)
    lp, _ = train_batch_grads(net, b, with_skill=with_skill,
                              with_act=with_act)
    np.copyto(net.params[j], P[j] - eps * d)
    lm, _ = train_batch_grads(net, b, with_skill=with_skill,
                              with_act=with_act)
    np.copyto(net.params[j], P[j])
    tot = (lambda L: L.get("ce", 0.0) + L.get("mse", 0.0))
    return (tot(lp) - tot(lm)) / (2 * eps), d

_, g_full = train_batch_grads(net, b, with_skill=True, with_act=True)
_, g_mse = train_batch_grads(net, b, with_skill=False, with_act=True)
P = [p.copy() for p in net.params]
# 主干输出层权重：只开 MSE 时与有限差分包一一对应
num, d = dir_deriv(net, P, 6, with_skill=False, with_act=True)
pred = float(np.sum(g_mse[6] * d))
rel = abs(pred - num) / max(abs(num), 1e-12)
check("主干/动作头梯度与有限差分一致", rel < 1e-5,
      f"相对误差 {rel:.1e}")
# ψ 的梯度经过 tanh + 共享 trunk（技能 CE 与动作 MSE 两路都汇进来）
num, d = dir_deriv(net, P, 0, with_skill=True, with_act=True)
pred = float(np.sum(g_full[0] * d))
rel2 = abs(pred - num) / max(abs(num), 1e-12)
check("槽编码器梯度（经 tanh 与共享路径）与有限差分一致", rel2 < 1e-5,
      f"相对误差 {rel2:.1e}")
check("类型属性模式互不相同", len({tuple(z) for z in world.z_types}) == T)
grid = _all_z()
cnt = np.bincount(np.argmax(grid @ world.W_g.T, axis=1), minlength=S - 1)
p = cnt / cnt.sum()
check("W_g 的 argmax 在属性模式上分布均衡", p.min() >= 0.12,
      f"min={p.min():.2f} max={p.max():.2f}")
check("环境项与 Bx 项同量级（都标定到 0.25/D_A）",
      abs(world.oracle_env_gap(*_gap_args(world)) - 0.25 / D_A) < 0.02,
      f"env={world.oracle_env_gap(*_gap_args(world)):.4f}")

# ------------------------------------------------------------ A 组


自校验：手写反向 vs 有限差分；世界不变量
  [ok] 主干/动作头梯度与有限差分一致（相对误差 4.5e-09）
  [ok] 槽编码器梯度（经 tanh 与共享路径）与有限差分一致（相对误差 1.6e-08）
  [ok] 类型属性模式互不相同
  [ok] W_g 的 argmax 在属性模式上分布均衡（min=0.19 max=0.25）
  [ok] 环境项与 Bx 项同量级（都标定到 0.25/D_A）（env=0.0310）


**B 组：环境数量的 scaling**

In [ ]:
banner("A 组：共训练配方消融（WD=web 数据 / ME=多环境 / CE=跨本体监督）")
rngP = np.random.default_rng(500)
pools = dict(mm=rngP.normal(size=(N_MM_ENV, D_CTX)),
             me=rngP.normal(size=(N_ME_ENV, D_CTX)),
             ce=rngP.normal(size=(20, D_CTX)),
             test=rngP.normal(size=(N_TEST_ENV, D_CTX)))
full = avg_over_seeds(world, seeds, pools)
no_wd = avg_over_seeds(world, seeds, pools, use_wd=False)
no_me = avg_over_seeds(world, seeds, pools, use_me=False)
no_ce = avg_over_seeds(world, seeds, pools, use_ce=False)
mm_only = avg_over_seeds(world, seeds, pools, use_wd=False, use_me=False,
                         use_ce=False)

def row(name, r):
    print(f"  {name:<10}  ID技能 {r['id'][0]:.3f}  复合 {r['oodc'][0]:.3f}"
          f"  web目标 {r['oodt'][0]:.3f}  新环境真值技能误差 {r['env'][2]:.3f}"
          f"  ID端到端误差 {r['id'][1]:.3f}")

for name, r in [("full", full), ("no WD", no_wd), ("no ME", no_me),
                ("no CE", no_ce), ("MM only", mm_only)]:
    row(name, r)

check("full 在 ID 技能选择上 ≥ 0.85", full["id"][0] > 0.85,
      f"{full['id'][0]:.3f}")
check("web 语义监督把未见类别上的技能选择抬起来（> +0.10）",
      full["oodt"][0] - no_wd["oodt"][0] > 0.10,
      f"{no_wd['oodt'][0]:.3f} → {full['oodt'][0]:.3f}")
check("WD 的收益集中在未见类别：相对降幅比 ID 大 0.15 以上",
      (1 - no_wd["oodt"][0] / full["oodt"][0])
      - (1 - no_wd["id"][0] / full["id"][0]) > 0.15,
      f"web目标相对降 {1 - no_wd['oodt'][0] / full['oodt'][0]:.2f}，"
      f"ID 相对降 {1 - no_wd['id'][0] / full['id'][0]:.2f}")
check("去掉 ME 后新环境动作变差（真值技能条件，> +0.03）",
      no_me["env"][2] - full["env"][2] > 0.03,
      f"{full['env'][2]:.3f} → {no_me['env'][2]:.3f}")
check("ME 的收益主要在新环境：新环境伤害 > ID 伤害 0.02 以上",
      (no_me["env"][2] - full["env"][2])
      - (no_me["id"][2] - full["id"][2]) > 0.02,
      f"新环境 +{no_me['env'][2] - full['env'][2]:.3f}，"
      f"ID +{no_me['id'][2] - full['id'][2]:.3f}")
check("CE 在探针里的边际收益很小：拿掉它对复合约束影响 < 0.12",
      abs(full["oodc"][0] - no_ce["oodc"][0]) < 0.12,
      f"{full['oodc'][0]:.3f} vs {no_ce['oodc'][0]:.3f}"
      "（探针未建模跨本体动作空间差异，见 README 的边界说明）")
check("全配方对 MM-only 的优势落在未见类别与新环境动作",
      full["oodt"][0] - mm_only["oodt"][0] > 0.07 and
      mm_only["env"][2] - full["env"][2] > 0.02,
      f"web目标 {mm_only['oodt'][0]:.3f}→{full['oodt'][0]:.3f}，"
      f"新环境误差 {mm_only['env'][2]:.3f}→{full['env'][2]:.3f}")
check("共训练的收益在分布外：ID 端到端误差并不优于 MM-only（差 < 0.08）",
      abs(full["id"][1] - mm_only["id"][1]) < 0.08,
      f"{full['id'][1]:.3f} vs {mm_only['id'][1]:.3f}")

# ------------------------------------------------------------ B 组


A 组：共训练配方消融（WD=web 数据 / ME=多环境 / CE=跨本体监督）
  full        ID技能 0.929  复合 0.451  web目标 0.687  新环境真值技能误差 0.168  ID端到端误差 0.271
  no WD       ID技能 0.946  复合 0.406  web目标 0.455  新环境真值技能误差 0.164  ID端到端误差 0.256
  no ME       ID技能 0.927  复合 0.483  web目标 0.667  新环境真值技能误差 0.199  ID端到端误差 0.273
  no CE       ID技能 0.941  复合 0.434  web目标 0.703  新环境真值技能误差 0.171  ID端到端误差 0.259
  MM only     ID技能 0.932  复合 0.420  web目标 0.487  新环境真值技能误差 0.196  ID端到端误差 0.268
  [ok] full 在 ID 技能选择上 ≥ 0.85（0.929）
  [ok] web 语义监督把未见类别上的技能选择抬起来（> +0.10）（0.455 → 0.687）
  [ok] WD 的收益集中在未见类别：相对降幅比 ID 大 0.15 以上（web目标相对降 0.34，ID 相对降 -0.02）
  [ok] 去掉 ME 后新环境动作变差（真值技能条件，> +0.03）（0.168 → 0.199）
  [ok] ME 的收益主要在新环境：新环境伤害 > ID 伤害 0.02 以上（新环境 +0.031，ID +0.001）
  [ok] CE 在探针里的边际收益很小：拿掉它对复合约束影响 < 0.12（0.451 vs 0.434（探针未建模跨本体动作空间差异，见 README 的边界说明））
  [ok] 全配方对 MM-only 的优势落在未见类别与新环境动作（web目标 0.487→0.687，新环境误差 0.196→0.168）
  [ok] 共训练的收益在分布外：ID 端到端误差并不优于 MM-only（差 < 0.08）（0.271 vs 0.268）


**C 组：joint / implicit HL / flat 对照**

In [ ]:
banner("B 组：环境数量的 scaling（完整配方，只改 MM+ME 的环境池大小）")
n_envs = [3, 6, 12, 25, 50, 100]
rngB = np.random.default_rng(202)
big_pool = rngB.normal(size=(106, D_CTX))
scal = {}
for n in n_envs:
    pool = dict(mm=big_pool[:min(n, N_MM_ENV)],
                me=big_pool[N_MM_ENV:n] if n > N_MM_ENV
                else big_pool[:0],
                ce=pools["ce"], test=pools["test"])
    r = avg_over_seeds(world, seeds, pool)
    # 对照：把测试环境也放进训练池（对应论文 Fig.8 的 green control）
    ctrl_mm = np.concatenate([pool["mm"], pools["test"]], axis=0)
    ctrl_me = np.concatenate([pool["me"], pools["test"]], axis=0) \
        if len(pool["me"]) else pools["test"]
    rc = avg_over_seeds(world, seeds,
                        dict(mm=ctrl_mm, me=ctrl_me, ce=pools["ce"],
                             test=pools["test"]))
    scal[n] = (r, rc)
    print(f"  N={n:>3}  新环境技能 {r['env'][0]:.3f}  "
          f"新环境真值技能误差 {r['env'][2]:.3f}   对照(含测试环境) "
          f"{rc['env'][0]:.3f} / {rc['env'][2]:.3f}")
lo, hi = scal[3][0], scal[100][0]
errs = [scal[n][0]["env"][2] for n in n_envs]
accs = [scal[n][0]["env"][0] for n in n_envs]
check("环境只影响「怎么做」不影响「做什么」：技能准确率对 N 几乎不动",
      max(accs) - min(accs) < 0.08,
      f"accs={[round(a, 3) for a in accs]}")
check("环境数 3→100：新环境动作误差好转 > 0.08",
      errs[0] - errs[-1] > 0.08, f"{errs[0]:.3f} → {errs[-1]:.3f}")
check("环境最少（N=3）是最差的一档", errs[0] == max(errs),
      f"errs={[round(e, 3) for e in errs]}")
check("N=100 已接近「直接用测试环境训练」的对照（误差差 < 0.04）",
      scal[100][1]["env"][2] - scal[100][0]["env"][2] < 0.04,
      f"{scal[100][0]['env'][2]:.3f} vs 对照 {scal[100][1]['env'][2]:.3f}")

# ------------------------------------------------------------ C 组


B 组：环境数量的 scaling（完整配方，只改 MM+ME 的环境池大小）
  N=  3  新环境技能 0.908  新环境真值技能误差 0.356   对照(含测试环境) 0.908 / 0.149
  N=  6  新环境技能 0.911  新环境真值技能误差 0.239   对照(含测试环境) 0.932 / 0.152
  N= 12  新环境技能 0.913  新环境真值技能误差 0.192   对照(含测试环境) 0.915 / 0.152
  N= 25  新环境技能 0.908  新环境真值技能误差 0.166   对照(含测试环境) 0.925 / 0.151
  N= 50  新环境技能 0.929  新环境真值技能误差 0.167   对照(含测试环境) 0.931 / 0.151
  N=100  新环境技能 0.917  新环境真值技能误差 0.166   对照(含测试环境) 0.918 / 0.152
  [ok] 环境只影响「怎么做」不影响「做什么」：技能准确率对 N 几乎不动（accs=[0.908, 0.911, 0.913, 0.908, 0.929, 0.917]）
  [ok] 环境数 3→100：新环境动作误差好转 > 0.08（0.356 → 0.166）
  [ok] 环境最少（N=3）是最差的一档（errs=[0.356, 0.239, 0.192, 0.166, 0.167, 0.166]）
  [ok] N=100 已接近「直接用测试环境训练」的对照（误差差 < 0.04）（0.166 vs 对照 0.152）


**D 组：状态进 prompt 的量化代价**

In [ ]:
banner("C 组：joint vs implicit HL vs flat（对应论文 Fig.13）")
# full 已是 joint
flat = avg_over_seeds(world, seeds, pools, use_skill_input=False,
                      train_skill=False)
implicit = avg_over_seeds(world, seeds, pools, use_skill_input=False,
                          train_skill=True)

def row2(name, r):
    print(f"  {name:<10}  ID端到端误差 {r['id'][1]:.3f}  "
          f"新环境端到端 {r['env'][1]:.3f}  复合端到端 {r['oodc'][1]:.3f}"
          f"  ID真值技能误差 {r['id'][2]:.3f}")

row2("joint", full)
row2("implicit", implicit)
row2("flat", flat)
check("两阶段推理在分布内更省：joint 的 ID 端到端误差比 flat 低 > 0.02",
      flat["id"][1] - full["id"][1] > 0.02,
      f"{full['id'][1]:.3f} vs {flat['id'][1]:.3f}")
check("joint 也比 implicit 低 > 0.05：只共训练不跑高层拿不到分布内收益",
      implicit["id"][1] - full["id"][1] > 0.05,
      f"{full['id'][1]:.3f} vs {implicit['id'][1]:.3f}")
check("只共训练不跑高层，分布内反而略差（技能头分走了容量）",
      implicit["id"][1] - flat["id"][1] > 0.01,
      f"flat {flat['id'][1]:.3f} vs implicit {implicit['id'][1]:.3f}")
check("分布外（未见复合约束）上离散接口成了单点故障：joint 比 flat 差 > 0.15",
      full["oodc"][1] - flat["oodc"][1] > 0.15,
      f"joint {full['oodc'][1]:.3f} vs flat {flat['oodc'][1]:.3f}")
check("implicit 落在 joint 与 flat 之间（复合约束端到端误差）",
      flat["oodc"][1] < implicit["oodc"][1] < full["oodc"][1],
      f"flat {flat['oodc'][1]:.3f} / implicit {implicit['oodc'][1]:.3f}"
      f" / joint {full['oodc'][1]:.3f}")
check("joint 在新环境动作上明显最好（真值技能条件，比 flat 低 > 0.10）",
      flat["env"][2] - full["env"][2] > 0.10,
      f"{full['env'][2]:.3f} vs {flat['env'][2]:.3f}")

# ------------------------------------------------------------ D 组


C 组：joint vs implicit HL vs flat（对应论文 Fig.13）
  joint       ID端到端误差 0.271  新环境端到端 0.267  复合端到端 1.159  ID真值技能误差 0.167
  implicit    ID端到端误差 0.344  新环境端到端 0.336  复合端到端 1.005  ID真值技能误差 0.344
  flat        ID端到端误差 0.303  新环境端到端 0.290  复合端到端 0.761  ID真值技能误差 0.303
  [ok] 两阶段推理在分布内更省：joint 的 ID 端到端误差比 flat 低 > 0.02（0.271 vs 0.303）
  [ok] joint 也比 implicit 低 > 0.05：只共训练不跑高层拿不到分布内收益（0.271 vs 0.344）
  [ok] 只共训练不跑高层，分布内反而略差（技能头分走了容量）（flat 0.303 vs implicit 0.344）
  [ok] 分布外（未见复合约束）上离散接口成了单点故障：joint 比 flat 差 > 0.15（joint 1.159 vs flat 0.761）
  [ok] implicit 落在 joint 与 flat 之间（复合约束端到端误差）（flat 0.761 / implicit 1.005 / joint 1.159）
  [ok] joint 在新环境动作上明显最好（真值技能条件，比 flat 低 > 0.10）（0.168 vs 0.290）


**汇总**

In [ ]:
banner("D 组：状态进 prompt 的量化代价（openpi 的 π0.5 格式）")
q = state_quant_probe()
print("  bins      相对误差（占噪声下界的倍数）   prompt token 数（18 维状态）")
for bins in (8, 16, 32, 64, 256):
    digits = len(str(bins - 1))
    tok = 18 * (digits + 1)
    print(f"  {bins:>4}      {q[bins]:>8.4f}  ({q[bins] / q['floor']:.2f}×)"
          f"                    {tok}")
check("量化误差随 bins 单调下降", all(
    q[a] > q[b] for a, b in zip((8, 16, 32, 64), (16, 32, 64, 256))))
check("量化从来不是瓶颈：粗到 8 bins 也只占噪声下界的 30% 不到",
      q[8] < 0.3 * q["floor"], f"{q[8]:.4f} vs floor {q['floor']:.4f}")
check("256 bins 的量化误差几乎为零（< 0.01× 噪声下界）",
      q[256] < 0.01 * q["floor"],
      f"{q[256]:.2e} vs floor {q['floor']:.4f}")
check("256 bins × 18 维 ≈ 72 个 token：π0 的 48 token 槽装不下，"
      "π0.5 的 200 token 槽才装得下", 48 < 18 * 4 < 200)

# ------------------------------------------------------------ 汇总


D 组：状态进 prompt 的量化代价（openpi 的 π0.5 格式）
  bins      相对误差（占噪声下界的倍数）   prompt token 数（18 维状态）
     8        0.0040  (0.03×)                    36
    16        0.0009  (0.01×)                    54
    32        0.0002  (0.00×)                    54
    64        0.0000  (0.00×)                    54
   256        0.0000  (0.00×)                    72
  [ok] 量化误差随 bins 单调下降
  [ok] 量化从来不是瓶颈：粗到 8 bins 也只占噪声下界的 30% 不到（0.0040 vs floor 0.1425）
  [ok] 256 bins 的量化误差几乎为零（< 0.01× 噪声下界）（3.03e-06 vs floor 0.1425）
  [ok] 256 bins × 18 维 ≈ 72 个 token：π0 的 48 token 槽装不下，π0.5 的 200 token 槽才装得下
